# Estudo de Caso 4.7 — Bombeamento em Prédio Residencial: Dimensionamento Hidráulico-Energético

| Campo | Detalhe |
|:------|:--------|
| **Capítulo Associado** | Capítulo 4 — Escoamento em Tubulações e Bombeamento |
| **Nível** | Graduação |
| **Tempo estimado** | 60 minutos |
| **Autor** | Jader Lugon Junior |
| **Método numérico** | Solução direta com correlações empíricas (Swamee–Jain) e análise de sensibilidade vetorizada |
| **Bibliografia** | Gobbi (s.d.), Cap. 2; Lugon Junior et al. (2026); White, *Mecânica dos Fluidos*, Cap. 6; NBR 5626 |

---

## 📋 Resumo

Este estudo de caso dimensiona o sistema de recalque de um prédio residencial de 12 pavimentos, aplicando a equação de energia generalizada entre os reservatórios inferior e superior. O fenômeno modelado é o escoamento permanente turbulento em tubulação de PVC, governado pela equação de Darcy–Weisbach com fator de atrito obtido pela correlação explícita de Swamee–Jain. O resultado esperado é a altura manométrica total ($H_B$), a potência da bomba e a verificação de cavitação via NPSH, com análise de sensibilidade à variação da vazão de projeto.

---

## 🎯 Objetivos de Aprendizagem

Ao final deste estudo de caso, o leitor será capaz de:

1. **Aplicar** a equação de energia generalizada entre dois reservatórios para o dimensionamento de sistemas de bombeamento.
2. **Calcular** perdas de carga distribuídas (Darcy–Weisbach) e localizadas (coeficiente $K$) em tubulações.
3. **Determinar** o fator de atrito de Darcy com a equação explícita de Swamee–Jain, classificando o regime pelo número de Reynolds.
4. **Dimensionar** a altura manométrica, a potência hidráulica e a potência elétrica da bomba considerando o rendimento global.
5. **Verificar** o NPSH disponível em relação ao NPSH requerido para prevenir cavitação.
6. **Analisar** a sensibilidade da altura manométrica e da potência à variação da vazão de projeto.
7. **Interpretar** o ponto de operação como interseção das curvas da bomba e do sistema.

> 📖 **Referência:** Gobbi (s.d.), Cap. 2, Seção 2.3; Lugon Junior et al. (2026).

## ⚙️ Seção 0: Configuração Inicial

Importação das bibliotecas científicas e definição do estilo de gráficos conforme o guia de padronização.

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown
import time

# Estilo padronizado de gráficos
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("=" * 60)
print("📦 CONFIGURAÇÃO DO AMBIENTE")
print("=" * 60)
print("  • numpy      : operações vetorizadas")
print("  • matplotlib : visualização padronizada")
print("  • pandas     : tabelas comparativas")
print("✅ Ambiente configurado com sucesso!")

## 🌊 Seção 1: Parâmetros do Sistema de Bombeamento

Um prédio residencial de 12 pavimentos necessita bombear água do reservatório inferior (subsolo) até o reservatório superior (cobertura).

| Parâmetro | Símbolo | Valor | Unidade |
|:---|:---:|:---:|:---:|
| Cota do reservatório inferior | $z_1$ | 0,0 | m |
| Cota do reservatório superior | $z_2$ | 40,0 | m |
| Diâmetro interno da tubulação (recalque) | $D$ | 50 | mm |
| Comprimento total da tubulação | $L$ | 55 | m |
| Vazão de projeto | $Q$ | 2,5 | L/s |
| Rugosidade absoluta do PVC | $\varepsilon$ | 0,0015 | mm |
| Rendimento global da bomba | $\eta$ | 70 | % |
| NPSH requerido pela bomba | $NPSH_{req}$ | 3,5 | m |

**Propriedades da água a 20 °C**

| Propriedade | Símbolo | Valor | Unidade |
|:---|:---:|:---:|:---:|
| Massa específica | $\rho$ | 998 | kg/m³ |
| Viscosidade cinemática | $\nu$ | 1,0 × 10⁻⁶ | m²/s |
| Peso específico | $\gamma$ | 9790 | N/m³ |
| Pressão de vapor | $p_{vap}$ | 2,34 | kPa |
| Pressão atmosférica | $p_{atm}$ | 101,325 | kPa |

**Acessórios do recalque (perdas localizadas)**

| Acessório | $K$ unitário | Quantidade |
|:---|:---:|:---:|
| Válvula de pé com crivo | 2,0 | 1 |
| Curva 90° padrão | 0,9 | 4 |
| Registro de globo aberto | 10,0 | 1 |
| Saída para reservatório | 1,0 | 1 |

In [ ]:
# ============================================================
# 1. PARÂMETROS DO SISTEMA E DO FLUIDO
# ============================================================
# --- Geometria e cotas ---
z1 = 0.0          # Cota do reservatório inferior [m]
z2 = 40.0         # Cota do reservatório superior [m]
D  = 0.050        # Diâmetro interno da tubulação de recalque [m]
L  = 55.0         # Comprimento total da tubulação [m]

# --- Condição de escoamento (forçamento) ---
Q = 0.0025        # Vazão volumétrica de projeto [m³/s] -> 2,5 L/s

# --- Propriedades da água a 20 °C ---
rho   = 998.0     # Massa específica [kg/m³]
nu    = 1.0e-6    # Viscosidade cinemática [m²/s]
gamma = 9790.0    # Peso específico [N/m³]
g     = 9.81      # Aceleração da gravidade [m/s²]
p_vap = 2340.0    # Pressão de vapor a 20 °C [Pa]
p_atm = 101325.0  # Pressão atmosférica [Pa]

# --- Rugosidade absoluta do PVC ---
epsilon = 1.5e-6  # Rugosidade absoluta [m] -> 0,0015 mm

# --- Acessórios do recalque (coeficientes K) ---
K_acessorios = {
    'Válvula de pé com crivo' : 2.0,
    'Curva 90° padrão (4 un)' : 4 * 0.9,
    'Registro de globo aberto': 10.0,
    'Saída para reservatório' : 1.0
}
K_total = sum(K_acessorios.values())   # Coeficiente total de perda localizada [-]

# --- Rendimento da bomba ---
eta = 0.70        # Rendimento global do conjunto motor-bomba [-]

print("=" * 60)
print("🌊 PARÂMETROS DO SISTEMA")
print("=" * 60)
print(f"  • Desnível geométrico : Δz = {z2 - z1:.1f} m")
print(f"  • Vazão de projeto    : Q  = {Q*1000:.1f} L/s")
print(f"  • Tubulação           : D  = {D*1000:.0f} mm, L = {L:.0f} m (PVC)")
print(f"  • Rugosidade relativa : ε/D = {epsilon/D:.2e}")
print(f"  • K total (localizada): {K_total:.1f}")
print(f"  • Rendimento          : η  = {eta*100:.0f}%")

## 🔬 Seção 2: Verificação / Validação

Antes do dimensionamento, validamos a consistência física dos parâmetros:

1. **Faixa de velocidade recomendada** para recalque predial (NBR 5626): tipicamente $0{,}6 \le V \le 2{,}4$ m/s.
2. **Consistência da continuidade**: $Q = V A$.
3. **Regime de escoamento**: o número de Reynolds deve confirmar escoamento turbulento, condição de validade da correlação de Swamee–Jain.

> 💡 **Observação:** a verificação prévia evita propagar dados inconsistentes e confirma a aplicabilidade das correlações empíricas escolhidas.

In [ ]:
# ============================================================
# 2. VERIFICAÇÃO PRÉVIA DOS PARÂMETROS
# ============================================================
A_verif = np.pi * D**2 / 4          # Área da seção transversal [m²]
V_verif = Q / A_verif               # Velocidade média [m/s]
Re_verif = V_verif * D / nu         # Número de Reynolds [-]

print("=" * 60)
print("🔬 VERIFICAÇÃO PRÉVIA")
print("=" * 60)
print(f"📏 Área da seção      : A  = {A_verif:.5e} m²")
print(f"📏 Velocidade média   : V  = {V_verif:.3f} m/s")
print(f"📏 Número de Reynolds : Re = {Re_verif:.2e}")

# Verificação 1: velocidade em faixa recomendada para recalque predial
assert 0.6 <= V_verif <= 2.4, f"⚠️ Velocidade {V_verif:.2f} m/s fora da faixa recomendada (0,6–2,4 m/s)"
print("✅ Verificação 1: velocidade dentro da faixa recomendada (0,6–2,4 m/s)")

# Verificação 2: continuidade Q = V*A
Q_recalc = V_verif * A_verif
erro_cont = abs(Q_recalc - Q) / Q * 100
assert erro_cont < 1e-9, "❌ Erro na continuidade"
print("✅ Verificação 2: equação da continuidade satisfeita")

# Verificação 3: regime turbulento (Re > 4000) -> Swamee-Jain aplicável
assert Re_verif > 4000, "❌ Regime não turbulento: Swamee-Jain pode não ser adequada"
print(f"✅ Verificação 3: regime turbulento confirmado (Re = {Re_verif:.2e} > 4000)")

print("\n✅ Todas as verificações prévias foram aprovadas!")

## 📐 Seção 3: Formulação Matemática

### 3.1 Velocidade média e número de Reynolds

Pela equação da continuidade e definição de Reynolds:

$$V = \frac{Q}{A} = \frac{4Q}{\pi D^2}, \qquad Re = \frac{V D}{\nu}$$

onde $V$ = velocidade média [m/s], $Q$ = vazão [m³/s], $A$ = área da seção [m²], $D$ = diâmetro [m], $\nu$ = viscosidade cinemática [m²/s].

### 3.2 Fator de atrito (Swamee–Jain)

Aproximação explícita da equação de Colebrook–White para regime turbulento:

$$f = \frac{0{,}25}{\left[\log_{10}\left(\dfrac{\varepsilon/D}{3{,}7} + \dfrac{5{,}74}{Re^{0,9}}\right)\right]^2}$$

onde $f$ = fator de atrito de Darcy [-], $\varepsilon$ = rugosidade absoluta [m].

### 3.3 Perdas de carga

Perda distribuída (Darcy–Weisbach) e perda localizada:

$$h_f = f\,\frac{L}{D}\,\frac{V^2}{2g}, \qquad h_s = K_{total}\,\frac{V^2}{2g}$$

onde $h_f$, $h_s$ = perdas de carga [m], $g$ = aceleração da gravidade [m/s²], $K_{total}$ = somatório dos coeficientes de perda localizada [-].

### 3.4 Equação de energia (altura manométrica)

Aplicando a equação de energia entre os dois reservatórios (ambos abertos à atmosfera, com $V_1 \approx V_2 \approx 0$):

$$H_B = (z_2 - z_1) + h_{f,total}$$

onde $H_B$ = altura manométrica da bomba [m].

### 3.5 Potência e NPSH

$$P_{hid} = \gamma\, Q\, H_B, \qquad P_{el} = \frac{P_{hid}}{\eta}$$

$$NPSH_d = \frac{p_{atm} - p_{vap}}{\gamma} + z_{suc} - h_{f,suc} - \frac{V_{suc}^2}{2g}$$

onde $P_{hid}$ e $P_{el}$ = potências hidráulica e elétrica [W], $NPSH_d$ = NPSH disponível [m], $z_{suc}$ = altura estática de sucção [m] (positiva para sucção afogada).

> 📖 **Referência:** Gobbi (s.d.), Cap. 2; White, Cap. 6.

## ⚡ Seção 4: Estabilidade / Critério Numérico

> 💡 **Observação:** neste caso, trata-se de **escoamento permanente** solucionado por correlações explícitas; portanto, **não se aplica a condição CFL** (própria de esquemas de marcha temporal em EDPs). Mantém-se esta seção com o critério físico de regime.

O critério de regime é dado pelo **número de Reynolds**:

$$Re = \frac{VD}{\nu}$$

| Faixa de $Re$ | Regime |
|:---:|:---|
| $Re < 2000$ | Laminar |
| $2000 \le Re \le 4000$ | Transição |
| $Re > 4000$ | Turbulento |

A correlação de Swamee–Jain é válida para $5\times10^3 < Re < 10^8$ e $10^{-6} < \varepsilon/D < 10^{-2}$, faixa que deve ser respeitada para garantir a precisão do fator de atrito.

## 🌪️ Seção 5: Forçamento / Condição de Entrada

O **forçamento** do sistema é a **vazão de projeto** $Q = 2{,}5$ L/s, imposta pela demanda do edifício. As **condições de contorno** são:

- **Montante (sucção):** pressão atmosférica no reservatório inferior ($p_1 = p_{atm}$), velocidade desprezível ($V_1 \approx 0$).
- **Jusante (recalque):** pressão atmosférica no reservatório superior ($p_2 = p_{atm}$), velocidade desprezível ($V_2 \approx 0$).

Essas condições justificam a simplificação da equação de energia, na qual os termos de pressão e de energia cinética se anulam entre os dois reservatórios.

## 🧮 Seção 6: Implementação Numérica

As funções abaixo encapsulam os cálculos hidráulicos, todas com docstrings no padrão NumPy e vetorizadas para permitir a análise de sensibilidade da Seção 9.

In [ ]:
# ============================================================
# 6.1 FUNÇÕES HIDRÁULICAS (vetorizadas)
# ============================================================
def fator_atrito_swamee_jain(Re, eps_D):
    """
    Calcula o fator de atrito de Darcy pela equação explícita de Swamee-Jain.

    Parameters
    ----------
    Re : float ou np.ndarray
        Número de Reynolds [-].
    eps_D : float
        Rugosidade relativa (epsilon/D) [-].

    Returns
    -------
    float ou np.ndarray
        Fator de atrito de Darcy f [-].

    Notes
    -----
    Aproximação explícita de Colebrook-White, válida para
    5e3 < Re < 1e8 e 1e-6 < eps_D < 1e-2.
    """
    return 0.25 / (np.log10(eps_D / 3.7 + 5.74 / Re**0.9))**2


def perda_carga_darcy(f, L, D, V, g=9.81):
    """
    Calcula a perda de carga distribuída pela equação de Darcy-Weisbach.

    Parameters
    ----------
    f : float ou np.ndarray
        Fator de atrito de Darcy [-].
    L : float
        Comprimento da tubulação [m].
    D : float
        Diâmetro interno [m].
    V : float ou np.ndarray
        Velocidade média [m/s].
    g : float, optional
        Aceleração da gravidade [m/s²].

    Returns
    -------
    float ou np.ndarray
        Perda de carga distribuída hf [m].
    """
    return f * (L / D) * (V**2 / (2 * g))


def perda_carga_localizada(K_total, V, g=9.81):
    """
    Calcula a perda de carga localizada (somatório dos coeficientes K).

    Parameters
    ----------
    K_total : float
        Somatório dos coeficientes de perda localizada [-].
    V : float ou np.ndarray
        Velocidade média [m/s].
    g : float, optional
        Aceleração da gravidade [m/s²].

    Returns
    -------
    float ou np.ndarray
        Perda de carga localizada hs [m].
    """
    return K_total * (V**2 / (2 * g))


def npsh_disponivel(p_atm, p_vap, gamma, z_suc, hf_suc, V_suc, g=9.81):
    """
    Calcula o NPSH disponível para sucção afogada.

    Parameters
    ----------
    p_atm : float
        Pressão atmosférica [Pa].
    p_vap : float
        Pressão de vapor do líquido [Pa].
    gamma : float
        Peso específico [N/m³].
    z_suc : float
        Altura estática de sucção [m] (positiva para sucção afogada).
    hf_suc : float
        Perda de carga total na sucção [m].
    V_suc : float
        Velocidade na tubulação de sucção [m/s].
    g : float, optional
        Aceleração da gravidade [m/s²].

    Returns
    -------
    float
        NPSH disponível [m].
    """
    return (p_atm - p_vap) / gamma + z_suc - hf_suc - V_suc**2 / (2 * g)


print("✅ Funções hidráulicas definidas com sucesso!")

In [ ]:
# ============================================================
# 6.2 CÁLCULO PRINCIPAL — RECALQUE
# ============================================================
t0 = time.time()

# Passo 1: velocidade média e número de Reynolds
A  = np.pi * D**2 / 4        # Área da seção transversal [m²]
V  = Q / A                   # Velocidade média [m/s]
Re = V * D / nu              # Número de Reynolds [-]

# Classificação do regime de escoamento
if Re < 2000:
    regime = "Laminar"
elif Re < 4000:
    regime = "Transição"
else:
    regime = "Turbulento"

# Passo 2: fator de atrito (Swamee-Jain)
epsilon_D = epsilon / D      # Rugosidade relativa [-]
f = fator_atrito_swamee_jain(Re, epsilon_D)

# Passo 3: perdas de carga
hf       = perda_carga_darcy(f, L, D, V, g)        # Perda distribuída [m]
hs       = perda_carga_localizada(K_total, V, g)   # Perda localizada [m]
hf_total = hf + hs                                 # Perda total [m]

# Passo 4: altura manométrica e potência
HB    = (z2 - z1) + hf_total      # Altura manométrica [m]
P_hid = gamma * Q * HB            # Potência hidráulica [W]
P_el  = P_hid / eta               # Potência elétrica [W]
P_cv  = P_el / 735.5              # Potência em cv (1 cv = 735,5 W)

tempo_calc = time.time() - t0

print("=" * 60)
print(" RESULTADOS DO DIMENSIONAMENTO — RECALQUE")
print("=" * 60)
print(f"\n📏 Hidráulica:")
print(f"  • Área da seção       : A  = {A:.5e} m²")
print(f"  • Velocidade média    : V  = {V:.3f} m/s")
print(f"  • Número de Reynolds  : Re = {Re:.2e} ({regime})")
print(f"  • Rugosidade relativa : ε/D = {epsilon_D:.2e}")
print(f"  • Fator de atrito     : f  = {f:.4f}")
print(f"\n📊 Perdas de carga:")
print(f"  • Perda distribuída   : hf = {hf:.2f} m")
print(f"  • Perda localizada    : hs = {hs:.2f} m")
print(f"  • Perda total         : h  = {hf_total:.2f} m")
print(f"\n🔋 Bomba:")
print(f"  • Altura manométrica  : HB = {HB:.2f} m")
print(f"  • Potência hidráulica : {P_hid/1000:.3f} kW")
print(f"  • Potência elétrica   : {P_el/1000:.3f} kW ({P_cv:.2f} cv)")
print(f"\n⏱️ Tempo de cálculo: {tempo_calc*1000:.2f} ms")

In [ ]:
# ============================================================
# 6.3 VERIFICAÇÃO DE CAVITAÇÃO — NPSH
# ============================================================
# Dados da sucção (bomba afogada, 1,5 m abaixo do nível do RI)
D_suc   = 0.075     # Diâmetro da tubulação de sucção [m]
L_suc   = 4.0       # Comprimento da tubulação de sucção [m]
z_suc   = 1.5       # Altura estática de sucção afogada [m]
K_suc   = 3.4       # Somatório de K na sucção (crivo + curva + entrada) [-]
NPSH_req = 3.5      # NPSH requerido pela bomba [m]

# Hidráulica da sucção
A_suc   = np.pi * D_suc**2 / 4                 # Área da sucção [m²]
V_suc   = Q / A_suc                            # Velocidade na sucção [m/s]
Re_suc  = V_suc * D_suc / nu                   # Reynolds na sucção [-]
f_suc   = fator_atrito_swamee_jain(Re_suc, epsilon / D_suc)
hf_suc  = perda_carga_darcy(f_suc, L_suc, D_suc, V_suc, g) + \
          perda_carga_localizada(K_suc, V_suc, g)          # Perda total na sucção [m]

# NPSH disponível
NPSH_d  = npsh_disponivel(p_atm, p_vap, gamma, z_suc, hf_suc, V_suc, g)
margem  = NPSH_d - NPSH_req

print("=" * 60)
print("⚠️ VERIFICAÇÃO DO NPSH")
print("=" * 60)
print(f"  • (p_atm - p_vap)/γ = {(p_atm - p_vap)/gamma:.2f} m")
print(f"  • Altura de sucção  = {z_suc:.2f} m (afogada)")
print(f"  • Perda na sucção   = {hf_suc:.3f} m")
print(f"  • V_suc²/(2g)       = {V_suc**2/(2*g):.3f} m")
print(f"\n  • NPSH disponível   : {NPSH_d:.2f} m")
print(f"  • NPSH requerido    : {NPSH_req:.2f} m")
print(f"  • Margem de segurança: {margem:.2f} m")

if margem > 0:
    print("\n✅ NPSH_d > NPSH_req → Sistema VIÁVEL (sem risco de cavitação)")
else:
    print("\n❌ NPSH_d < NPSH_req → Sistema INVIÁVEL (risco de cavitação)")

## 📊 Seção 7: Visualização

In [ ]:
# ============================================================
# 7.1 COMPOSIÇÃO DA ALTURA MANOMÉTRICA
# ============================================================
labels = ['Desnível Geométrico\n(Δz)', 'Perda Distribuída\n(hf)', 'Perda Localizada\n(hs)']
values = [z2 - z1, hf, hs]
colors = ['#4C72B0', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(labels, values, color=colors, edgecolor='black', linewidth=1.2)
ax.set_ylabel('Altura [m]', fontsize=12)
ax.set_title('Composição da Altura Manométrica Total ($H_B$)', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(values) * 1.25)

# Rótulos sobre as barras
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.5, f'{yval:.2f} m',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.grid(True, axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

> 💡 **Observação:** o desnível geométrico domina amplamente a altura manométrica (40 m de um total de 43,17 m). As perdas de carga (distribuída + localizada) somam apenas ~3,2 m, o que confirma que o diâmetro de 50 mm é adequado para a vazão de projeto, mantendo as perdas em patamar razoável (~7% do total).

In [ ]:
# ============================================================
# 7.2 CURVA DA BOMBA × CURVA DO SISTEMA
# ============================================================
# Curva da bomba: H = H0 - a*Q² (shutoff 30% acima do ponto de operação)
H0 = HB * 1.3                       # Altura a Q = 0 [m]
a_bomba = (H0 - HB) / Q**2          # Coeficiente da curva da bomba [s²/m⁵]

# Curva do sistema: H = Δz + K_sys*Q²
K_sistema = hf_total / Q**2         # Resistência do sistema [s²/m⁵]

# Varredura de vazões
Q_curve = np.linspace(0, Q * 1.5, 100)     # Vazões de 0 a 3,75 L/s [m³/s]
H_bomba = H0 - a_bomba * Q_curve**2        # Curva da bomba [m]
H_sistema = (z2 - z1) + K_sistema * Q_curve**2  # Curva do sistema [m]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(Q_curve * 1000, H_bomba, 'b-', linewidth=2, label='Curva da bomba')
ax.plot(Q_curve * 1000, H_sistema, 'r-', linewidth=2, label='Curva do sistema')
ax.plot(Q * 1000, HB, 'ko', markersize=10,
        label=f'Ponto de operação (Q = {Q*1000:.1f} L/s, H = {HB:.1f} m)')
ax.set_xlabel('Vazão, Q [L/s]', fontsize=12)
ax.set_ylabel('Altura manométrica, H [m]', fontsize=12)
ax.set_title('Curva Característica da Bomba e do Sistema', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

> 💡 **Observação:** o ponto de operação é a interseção entre a curva da bomba e a curva do sistema. Para alterar a vazão, pode-se: (i) estrangulamento por válvula — desloca a curva do sistema; ou (ii) variador de frequência — desloca a curva da bomba, solução energeticamente mais eficiente.

## 📈 Seção 8: Análise Quantitativa

Consolidação dos resultados em tabela comparativa e métricas de desempenho.

In [ ]:
# ============================================================
# 8.1 TABELA DE RESULTADOS
# ============================================================
resultados = pd.DataFrame({
    'Parâmetro': ['Velocidade média (V)', 'Número de Reynolds (Re)',
                  'Fator de atrito (f)', 'Perda distribuída (hf)',
                  'Perda localizada (hs)', 'Perda total (hf,total)',
                  'Altura manométrica (HB)', 'Potência hidráulica',
                  'Potência elétrica', 'NPSH disponível',
                  'Margem NPSH'],
    'Valor': [f'{V:.3f} m/s', f'{Re:.2e} ({regime})', f'{f:.4f}',
              f'{hf:.2f} m', f'{hs:.2f} m', f'{hf_total:.2f} m',
              f'{HB:.2f} m', f'{P_hid/1000:.3f} kW',
              f'{P_el/1000:.3f} kW ({P_cv:.2f} cv)', f'{NPSH_d:.2f} m',
              f'{margem:.2f} m']
})

display(Markdown(resultados.to_markdown(index=False)))

print("\n" + "=" * 60)
print("📈 MÉTRICAS DE DESEMPENHO")
print("=" * 60)
print(f"  • Percentual de perdas sobre HB : {hf_total/HB*100:.1f}%")
print(f"  • Energia específica            : {g*HB:.1f} J/kg")
print(f"  • Tempo total de cálculo        : {tempo_calc*1000:.2f} ms")
print(f"\n✅ Análise quantitativa concluída!")

## 🔍 Seção 9: Sensibilidade / Convergência

A perda de carga varia aproximadamente com o quadrado da vazão ($h_f \propto Q^2$). Analisamos como a altura manométrica e a potência elétrica se comportam quando a vazão de projeto varia entre 1,0 L/s e 5,0 L/s. Todo o cálculo é **vetorizado** (sem loops Python).

In [ ]:
# ============================================================
# 9.1 ANÁLISE DE SENSIBILIDADE À VAZÃO (vetorizada)
# ============================================================
Q_range = np.linspace(0.001, 0.005, 200)    # Vazões de 1 a 5 L/s [m³/s]
V_range = Q_range / A                        # Velocidade [m/s]
Re_range = V_range * D / nu                  # Reynolds [-]
f_range = fator_atrito_swamee_jain(Re_range, epsilon_D)

hf_range = perda_carga_darcy(f_range, L, D, V_range, g)   # Perda distribuída [m]
hs_range = perda_carga_localizada(K_total, V_range, g)    # Perda localizada [m]
HB_range = (z2 - z1) + hf_range + hs_range                # Altura manométrica [m]
P_el_range = (gamma * Q_range * HB_range / eta) / 1000    # Potência elétrica [kW]

# Tabela comparativa em pontos selecionados
pontos = [0.001, 0.0025, 0.005]
tabela_sens = pd.DataFrame({
    'Q [L/s]'      : [q*1000 for q in pontos],
    'HB [m]'       : [(z2 - z1) + perda_carga_darcy(fator_atrito_swamee_jain((q/A)*D/nu, epsilon_D), L, D, q/A, g) + perda_carga_localizada(K_total, q/A, g) for q in pontos],
    'P_el [kW]'    : [gamma*q*((z2 - z1) + perda_carga_darcy(fator_atrito_swamee_jain((q/A)*D/nu, epsilon_D), L, D, q/A, g) + perda_carga_localizada(K_total, q/A, g))/eta/1000 for q in pontos]
})
tabela_sens = tabela_sens.round(2)
display(Markdown(tabela_sens.to_markdown(index=False)))

# Gráficos de sensibilidade
fig, axs = plt.subplots(1, 2, figsize=(14, 6))

axs[0].plot(Q_range * 1000, HB_range, 'b-', linewidth=2)
axs[0].axvline(x=2.5, color='r', linestyle='--', label='Projeto (2,5 L/s)')
axs[0].set_xlabel('Vazão Q [L/s]', fontsize=12)
axs[0].set_ylabel('Altura manométrica $H_B$ [m]', fontsize=12)
axs[0].set_title('$H_B$ em função da Vazão', fontsize=14, fontweight='bold')
axs[0].legend(fontsize=11)
axs[0].grid(True, alpha=0.3)

axs[1].plot(Q_range * 1000, P_el_range, 'r-', linewidth=2)
axs[1].axvline(x=2.5, color='b', linestyle='--', label='Projeto (2,5 L/s)')
axs[1].set_xlabel('Vazão Q [L/s]', fontsize=12)
axs[1].set_ylabel('Potência elétrica [kW]', fontsize=12)
axs[1].set_title('Potência em função da Vazão', fontsize=14, fontweight='bold')
axs[1].legend(fontsize=11)
axs[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Insight: se a vazão dobrar para 5,0 L/s, a perda de carga total")
print("   aumenta ~4×, exigindo bomba mais potente e elevando o custo")
print("   operacional de energia de forma não linear.")

> 💡 **Observação:** a sensibilidade confirma o comportamento quadrático das perdas de carga. A potência cresce de forma ainda mais acentuada (produto $Q \times H_B$), evidenciando a importância de dimensionar a vazão de projeto com critério para evitar sobredimensionamento energético.

## 💡 Seção 10: Discussão

### Interpretação física dos resultados

1. **Domínio do desnível geométrico:** com 40 m de desnível contra ~3,2 m de perdas, a altura estática é o componente dominante da altura manométrica. Isso é típico de recalques prediais elevados, onde a escolha da bomba é regida principalmente pela altura geométrica.
2. **Regime turbulento plenamente desenvolvido:** o número de Reynolds (~6,4 × 10⁴) situa o escoamento em regime turbulento, validando o uso da correlação de Swamee–Jain e justificando o fator de atrito obtido (~0,0198).
3. **Margem confortável de NPSH:** a sucção afogada (1,5 m) garante NPSH disponível (~11 m) muito acima do requerido (3,5 m), eliminando risco de cavitação — condição essencial para a vida útil da bomba.

### Limitações do modelo

- **Escoamento permanente idealizado:** não considera transientes (partida/parada, golpe de aríete), que podem gerar sobrepressões significativas.
- **Coeficientes K tabelados:** os valores de perda localizada são médios de literatura e podem variar conforme o fabricante e o desgaste dos acessórios.
- **Rendimento constante:** admite-se $\eta = 70\%$ fixo, quando na prática o rendimento varia com o ponto de operação da curva da bomba.
- **Rugosidade constante:** não considera o envelhecimento/incrustação da tubulação ao longo dos anos.

### Aplicações práticas

- **Seleção de bombas comerciais:** a altura manométrica e a vazão de projeto permitem escolher a bomba em catálogo, aplicando margem de segurança de 10–15%.
- **Estimativa de custo operacional:** a potência elétrica calculada fundamenta o dimensionamento elétrico e a previsão de consumo energético do edifício.

### Recomendações de engenharia

1. Selecionar bomba comercial com $H_B \approx 45\text{–}50$ m e $Q = 2{,}5$ L/s.
2. Verificar o NPSH disponível na sucção em campo para evitar cavitação.
3. Considerar margem de segurança de 10–15% na altura manométrica.
4. Instalar válvula de retenção na saída da bomba para evitar golpe de aríete e retorno da água ao desligar.

### Conexão com a teoria

O caso aplica diretamente a conservação de energia (equação de Bernoulli estendida) e os conceitos de perda de carga apresentados no Capítulo 4, conectando a formulação de Darcy–Weisbach e a correlação de Swamee–Jain ao dimensionamento real de sistemas de bombeamento.

> 📖 **Referência:** Gobbi (s.d.), Cap. 2; Lugon Junior et al. (2026).

---

## 🔄 Seção 11: Navegação

- [📚 Voltar ao Capítulo 4](../notebooks/04_Escoamento_Tubulacoes.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">
Estudo de Caso 4.7 — EC_04_07_Bombeamento_Predio_Residencial.ipynb<br>
Parte da coleção <i>"Fenômenos de Transporte: Fundamentos e Modelagem Computacional"</i>
</div>

In [ ]:
# ============================================================
# ENCERRAMENTO
# ============================================================
print("=" * 60)
print("✅ ESTUDO DE CASO 4.7 CONCLUÍDO COM SUCESSO!")
print("=" * 60)
print("\n🎓 Você completou o dimensionamento do sistema de bombeamento.")
print("   Resultados principais:")
print(f"   • Altura manométrica : HB = {HB:.2f} m")
print(f"   • Potência elétrica  : {P_el/1000:.2f} kW ({P_cv:.2f} cv)")
print(f"   • NPSH disponível    : {NPSH_d:.2f} m (margem {margem:.2f} m)")
print("\n💡 Próximo passo: explore o Estudo de Caso 5.1 (Hidrodinâmica do Rio Macaé).")